In [ ]:
# Load packages
import os
import hail as hl
import pandas as pd 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

In [ ]:
# Run this cell once
hl.init(
    gcs_requester_pays_configuration=os.environ['GOOGLE_CLOUD_PROJECT']
)

In [ ]:
# Load packages 
import os
import subprocess
import pandas as pd

# Store all sample-variant exclusion pairs
remove_pairs = set()

# Define path to workspace bucket 
bucket = os.environ["WORKSPACE_BUCKET"]

# GCS source and local destination
gcs_path = f"{bucket}/data/chip/remove.txt"
local_path = f"{os.getcwd()}/remove.txt"

# Download from Google Cloud Storage
subprocess.run(
    ["gcloud", "storage", "cp", gcs_path, local_path],
    check=True
)

# Read the downloaded file
remove_df = pd.read_csv(
    local_path,
    sep="\t",
    dtype=str
)

# Create sample-variant pairs
remove_pairs = set(
    remove_df["sample"] + "|" + remove_df["variant"]
)

print("Total unique exclusions across chromosomes:", len(remove_pairs))

In [ ]:

# Iterate through the chromosomes requiring genotype-level quality-control filtering
for chrom in [2, 4, 15, 17]:

    # Define the input PLINK file prefix for the current chromosome
    prefix = (
        f"{os.environ['WORKSPACE_BUCKET']}/data/chip/"
        f"plink/EUR_passed_chr{chrom}"
    )

    # 1. Import the chromosome-specific PLINK dataset into a Hail MatrixTable
    mt = hl.import_plink(
        bed=f"{prefix}.bed",
        bim=f"{prefix}.bim",
        fam=f"{prefix}.fam"
    )

    # 2. Identify variant-sample combinations requiring genotype-level filtering
    # Convert the removal list into a Hail set for efficient membership checking.
    # Each identifier combines the sample ID and variant ID, separated by '|'.
    remove_pairs_hl = hl.literal(remove_pairs)

    # Annotate genotype entries matching the removal list
    mt = mt.annotate_entries(
        _target=remove_pairs_hl.contains(mt.s + "|" + mt.rsid)
    )

     # Count the number of genotype entries matching the removal criteria
    print("Matching calls:", mt.aggregate_entries(
        hl.agg.count_where(mt._target)
    ))

    # 3. Mask genotypes corresponding to identified variant-sample combinations
    # Set matching genotype calls to missing while retaining all other entries
    mt = mt.annotate_entries(
        GT=hl.if_else(mt._target, hl.missing(hl.tcall), mt.GT)
    )

    # 4. Validate genotype masking
    # Confirm that no targeted genotype calls remain defined
    remaining = mt.aggregate_entries(
        hl.agg.count_where(mt._target & hl.is_defined(mt.GT))
    )
    print("Target calls still defined (expected 0):", remaining)
    
    # Stop execution if any targeted genotype calls remain unmasked
    assert remaining == 0

    # 5. Export the updated MatrixTable back into PLINK format
    # The output dataset retains the original variants and samples,
    # with identified genotype calls masked as missing
    hl.export_plink(
        mt,
        f"{os.environ['WORKSPACE_BUCKET']}/data/chip/"
        f"plink/EUR_ab0.35_dp20_chr{chrom}"
    )